[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/junlinghu/book-agents/blob/main/tutorial/2-data-and-files.ipynb)

## Colab setup

On Google Colab, use the setup in `tutorial/common/colab.py`. A fresh Colab runtime does not have this repository, so clone the repo into the session first. Then, before the lesson cells, run `setup_colab` from that module (import it from `tutorial.common.colab` and call it, or open the file and run `setup_colab` from there). The helper installs the tutorial packages, checks the repo out under `/content/book-agents` when it is still missing, and copies a Colab secret named `OPENAI_API_KEY`. Local Jupyter and VS Code can skip it when you already have the repo.

An API key is required. Set `OPENAI_API_KEY` in Colab secrets (the key icon, secret name `OPENAI_API_KEY`) before you run `setup_colab`. Locally, put the same name in the repository-root `.env`. A missing key stops the notebook. The key is not printed.

Edits you make in Colab stay in that session. They do not push to GitHub.

# 2. Shop data and files

Tutorial 1 returned one policy section. This notebook keeps that tool and adds two sensors the counter actually needs.

`read_shop_file` reads `policy.md` or `faq.md` from `tutorial/docs`. The path check is `tutorial.common.tools.read_file`. `query_inventory` reads the shelf seeded in this notebook from SQLite. The model passes a filter, not a SQL string.

The run asks which items are low and whether oat milk can ship. Both answers have to come from tools.

Shared helpers this lesson needs are imported, not copied from earlier notebooks. You can run this file by itself.

## Setup

This notebook calls the OpenAI Chat Completions API. `OPENAI_API_KEY` is required. Locally the key is loaded from the repository-root `.env` by `tutorial/common/client.py`. On Colab, `setup_colab` in `tutorial/common/colab.py` reads a secret of the same name. The value is not printed. A missing key stops the run.

Companion notes: `2-data-and-files.md`.

In [ ]:
import sys
from pathlib import Path


def find_root():
    """Repo root, whether the kernel started here or in tutorial."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        marker = candidate / "tutorial" / "common" / "client.py"
        if marker.is_file():
            return candidate
    raise RuntimeError(
        "Open this notebook inside the book-agents repository. "
        "The setup cell looks for tutorial/common/client.py."
    )


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tutorial.common.client import describe_runtime
from tutorial.runtime import chat, require_key

LESSON = '2-data-and-files'
LESSON_NUMBER = 2
require_key()
print("tutorial:", LESSON)
print(describe_runtime())


## Shared pieces

These imports are the earlier pieces this lesson uses. They come from `tutorial.common` and `tutorial.runtime`. You do not need to run the previous notebook first.


In [ ]:
import json

from tutorial.common.facts import get_shop_fact  # registers the tutorial 1 tool
from tutorial.common.harness import (
    TOOLS,
    assistant_message,
    call_tool,
    check,
    preview,
    system_text,
)
from tutorial.runtime import chat


## Tutorial 2 — files and the shelf

`read_shop_file` uses the path jail in `tutorial/common/tools.py`. `query_inventory` reads the shelf seeded in this notebook. The SQL stays in the function.

In [ ]:
# Shelf seed for this tutorial. The model never writes the SQL.
# sku, name, category, unit, reorder_point, on_hand, par.
import json
import sqlite3

from tutorial.common.harness import read_docs, register

SEED_ROWS = (
    ("HB-12", "House blend 12oz", "coffee", "bag", 6, 4, 18),
    ("HB-2LB", "House blend 2lb", "coffee", "bag", 4, 9, 10),
    ("ES-1KG", "Espresso beans 1kg", "coffee", "bag", 5, 5, 12),
    ("OM-32", "Oat milk 32oz", "dairy", "carton", 8, 3, 16),
    ("MLK-1", "Whole milk gallon", "dairy", "gallon", 4, 11, 12),
    ("ALM-1", "Almond meal", "bakery", "bag", 2, 1, 4),
    ("FL-01", "Paper filters", "supply", "box", 2, 7, 8),
)

STATE = {"conn": None}


def gap_for(row):
    """Cartons or bags to buy to reach par. The supplier note does not get a vote."""
    return int(row["par"]) - int(row["on_hand"])


def reset_db():
    if STATE["conn"] is not None:
        STATE["conn"].close()
    conn = sqlite3.connect(":memory:")
    conn.row_factory = sqlite3.Row
    conn.executescript(
        """
        CREATE TABLE products (
            sku TEXT PRIMARY KEY,
            name TEXT NOT NULL,
            category TEXT NOT NULL,
            unit TEXT NOT NULL,
            reorder_point INTEGER NOT NULL
        );
        CREATE TABLE inventory (
            sku TEXT PRIMARY KEY,
            on_hand INTEGER NOT NULL,
            par INTEGER NOT NULL
        );
        """
    )
    for sku, name, category, unit, reorder_point, on_hand, par in SEED_ROWS:
        conn.execute(
            "INSERT INTO products (sku, name, category, unit, reorder_point) VALUES (?, ?, ?, ?, ?)",
            (sku, name, category, unit, reorder_point),
        )
        conn.execute(
            "INSERT INTO inventory (sku, on_hand, par) VALUES (?, ?, ?)",
            (sku, on_hand, par),
        )
    conn.commit()
    STATE["conn"] = conn


def row_for(sku):
    if STATE["conn"] is None:
        reset_db()
    found = STATE["conn"].execute(
        """
        SELECT p.sku, p.name, p.category, p.unit, p.reorder_point, i.on_hand, i.par
        FROM products p
        JOIN inventory i ON p.sku = i.sku
        WHERE p.sku = ?
        """,
        (sku,),
    ).fetchone()
    if found is None:
        return None
    item = dict(found)
    item["gap"] = gap_for(item)
    return item


def query_inventory(args):
    """Read the shelf. The SQL stays in this function. The model passes filters, not a statement."""
    if STATE["conn"] is None:
        reset_db()
    only_low = bool(args.get("only_low", False))
    sku = args.get("sku")
    sql = (
        "SELECT p.sku, p.name, p.category, p.unit, p.reorder_point, i.on_hand, i.par "
        "FROM products p JOIN inventory i ON p.sku = i.sku"
    )
    params = []
    where = []
    if isinstance(sku, str) and sku.strip():
        where.append("p.sku = ?")
        params.append(sku.strip())
    if only_low:
        where.append("i.on_hand <= p.reorder_point")
    if where:
        sql += " WHERE " + " AND ".join(where)
    sql += " ORDER BY p.sku"
    rows = []
    for found in STATE["conn"].execute(sql, params):
        item = dict(found)
        item["gap"] = gap_for(item)
        rows.append(item)
    payload = {"rows": rows}
    if isinstance(sku, str) and sku.strip() and not rows:
        payload["error"] = "unknown sku " + sku.strip()
    return json.dumps(payload)


def read_shop_file(args):
    """Model-facing file tool. Only docs/policy.md and docs/faq.md resolve."""
    return read_docs(args.get("path", ""))


register(
    "read_shop_file",
    "Read a shop document. path is policy.md or faq.md. "
    "policy.md has returns, shipping, damage, and local delivery. "
    "faq.md has hours, the menu, and allergens. "
    "Paths outside docs/ return ERROR.",
    {"path": {"type": "string", "description": "policy.md or faq.md"}},
    ["path"],
    read_shop_file,
)
register(
    "query_inventory",
    "Read the café shelf from SQLite. Low stock means on_hand <= reorder_point. "
    "Each row includes gap, which is par minus on_hand. This tool does not write.",
    {
        "only_low": {"type": "boolean", "description": "True to keep only low or at-reorder rows."},
        "sku": {"type": "string", "description": "Optional sku, such as OM-32."},
    },
    [],
    query_inventory,
)


## Run

`system_text()` mentions only the tools imported above. Run this cell after the ones above. Each model turn calls the Chat Completions API. A check prints a warning when the wording differs from the expected trace, and the notebook continues.

In [ ]:
def run_exchange(user_text):
    """One model call, tool results, second model call. Tutorial 3 wraps this in a loop."""
    messages = [
        {"role": "system", "content": system_text()},
        {"role": "user", "content": user_text},
    ]
    print("QUESTION:", user_text)
    print()
    first = chat(messages, TOOLS)
    if not first["tool_calls"]:
        print("ANSWER:", first["content"])
        print("stop: final (no tool call)")
        return first["content"] or "", []
    messages.append(assistant_message(first))
    log = []
    for call in first["tool_calls"]:
        result = call_tool(call)
        line = call["name"] + " " + json.dumps(call["arguments"], sort_keys=True)
        log.append(line)
        print("[tool]", line)
        print(preview(result))
        print()
        messages.append({
            "role": "tool",
            "tool_call_id": call["id"],
            "content": result,
        })
    second = chat(messages, TOOLS)
    print("ANSWER:", second["content"])
    print("stop: final")
    return second["content"] or "", log


reset_db()
print("=== shelf and policy ===")
answer, log = run_exchange(
    "Which products are at or below their reorder point, and may we ship oat milk?",
)
check(any(line.startswith("query_inventory") for line in log), "inventory tool ran")
check(any(line.startswith("read_shop_file") for line in log), "policy file was read")
check("does not ship" in answer.lower(), "answer uses the shipping rule from the file")
check("HB-12" in answer and "OM-32" in answer, "low-stock skus are named")

print("=== carried forward: one shop fact ===")
fact, fact_log = run_exchange(
    "Can a customer return an opened bag of house coffee?",
)
check(any(line.startswith("get_shop_fact") for line in fact_log), "the lesson 1 tool still runs")
check("final sale" in fact.lower(), "the return rule still comes from the tool")
